# Notebook 01 : Data Collection
### YouTube Data API v3 Pull + Google Trends

**Purpose:** Pull real video-level data for 6 VALORANT creators (Q3 + Q4 2024) using the YouTube Data API v3, then fetch Google Trends context data via `pytrends`.

**Data already committed** in `data/raw/` and set `REFRESH_DATA = True` to re-pull live data.

In [ ]:
import pandas as pd
import json
import os
from datetime import datetime

# ── Config ──────────────────────────────────────────────────────────────────
REFRESH_DATA = False          # Set True to re-pull from APIs
API_KEY      = 'YOUR_KEY'    # YouTube Data API v3 key
DATA_DIR     = '../data/raw'

# Creators tracked
CREATORS = {
    'TenZ':             'UCckPYr9b_iVucz8ID1Q67sw',
    'tarik':            'UCa5f6pDiLMurRuQeWN0UMvQ',
    'Kyedae':           'UCjN_5A9I3KbFiS8zTJnmB_A',
    'aceu':             'UCQNFbfLnRW3sANqpONnMoFQ',
    'Sinatraa':         'UC8kBgRMJBlENmAE7PTqJbXw',
    'Protatomonster':   'UCzH3iADRIq1IJlIAsa4H0_Q',
}

print('Creators:', list(CREATORS.keys()))
print('REFRESH_DATA:', REFRESH_DATA)

## 1. YouTube Data API v3 — Pull Functions

Three endpoints used:
1. `channels` — get subscriber count + channel stats
2. `search` — list video IDs published in date range
3. `videos` — get full stats (views, likes, comments, duration) per video ID

In [ ]:
import urllib.request
import urllib.parse

BASE = 'https://www.googleapis.com/youtube/v3'

def yt_get(endpoint, params):
    """Minimal YouTube API GET wrapper."""
    params['key'] = API_KEY
    url = f"{BASE}/{endpoint}?{urllib.parse.urlencode(params)}"
    with urllib.request.urlopen(url) as r:
        return json.loads(r.read())

def get_channel_stats(channel_id):
    """Return dict with subscriberCount, viewCount, videoCount."""
    data = yt_get('channels', {
        'part': 'statistics',
        'id': channel_id
    })
    s = data['items'][0]['statistics']
    return {
        'subscribers':    int(s.get('subscriberCount', 0)),
        'lifetime_views': int(s.get('viewCount', 0)),
        'video_count':    int(s.get('videoCount', 0)),
    }

def search_videos(channel_id, published_after, published_before, max_pages=5):
    """Return list of video IDs published in date range."""
    # Quota cost: 100 units per page
    video_ids = []
    params = {
        'part':             'id',
        'channelId':        channel_id,
        'publishedAfter':   published_after,
        'publishedBefore':  published_before,
        'type':             'video',
        'maxResults':       50,
        'order':            'date',
    }
    for _ in range(max_pages):
        data = yt_get('search', params)
        video_ids += [item['id']['videoId'] for item in data.get('items', [])]
        token = data.get('nextPageToken')
        if not token:
            break
        params['pageToken'] = token
    return video_ids

def get_video_stats(video_ids):
    """Return list of dicts with view/like/comment/duration per video."""
    # Quota cost: 1 unit per 50 videos
    rows = []
    for i in range(0, len(video_ids), 50):
        chunk = video_ids[i:i+50]
        data = yt_get('videos', {
            'part':  'statistics,contentDetails,snippet',
            'id':    ','.join(chunk),
        })
        for item in data.get('items', []):
            stats = item.get('statistics', {})
            rows.append({
                'video_id':       item['id'],
                'title':          item['snippet']['title'],
                'published_date': item['snippet']['publishedAt'][:10],
                'views':          int(stats.get('viewCount',    0)),
                'likes':          int(stats.get('likeCount',    0)),
                'comments':       int(stats.get('commentCount', 0)),
                'duration':       item['contentDetails']['duration'],
            })
    return rows

print('API functions defined.')

## 2. Pull Q3 2024 Data (Jul 1 – Sep 30)

In [ ]:
Q3_AFTER  = '2024-07-01T00:00:00Z'
Q3_BEFORE = '2024-09-30T23:59:59Z'
Q4_AFTER  = '2024-10-01T00:00:00Z'
Q4_BEFORE = '2024-12-31T23:59:59Z'

if REFRESH_DATA:
    all_rows_q3 = []
    for creator, channel_id in CREATORS.items():
        print(f'Pulling Q3 for {creator}...')
        ch   = get_channel_stats(channel_id)
        vids = search_videos(channel_id, Q3_AFTER, Q3_BEFORE)
        rows = get_video_stats(vids)
        for r in rows:
            r['creator']     = creator
            r['subscribers'] = ch['subscribers']
        all_rows_q3.extend(rows)
        print(f'  {len(rows)} videos')

    df_q3 = pd.DataFrame(all_rows_q3)
    df_q3['month']       = df_q3['published_date'].str[:7]
    df_q3['engagements'] = df_q3['likes'] + df_q3['comments']
    df_q3['engagement_rate'] = (df_q3['engagements'] / df_q3['views'].replace(0,1) * 100).round(4)
    df_q3.to_csv(f'{DATA_DIR}/q3_2024_videos.csv', index=False)
    print(f'Saved {len(df_q3)} Q3 rows.')
else:
    df_q3 = pd.read_csv(f'{DATA_DIR}/q3_2024_videos.csv')
    print(f'Loaded {len(df_q3)} Q3 rows from file.')

df_q3.head(3)

## 3. Pull Q4 2024 Data (Oct 1 – Dec 31)

In [ ]:
if REFRESH_DATA:
    all_rows_q4 = []
    for creator, channel_id in CREATORS.items():
        print(f'Pulling Q4 for {creator}...')
        ch   = get_channel_stats(channel_id)
        vids = search_videos(channel_id, Q4_AFTER, Q4_BEFORE)
        rows = get_video_stats(vids)
        for r in rows:
            r['creator']     = creator
            r['subscribers'] = ch['subscribers']
        all_rows_q4.extend(rows)
        print(f'  {len(rows)} videos')

    df_q4 = pd.DataFrame(all_rows_q4)
    df_q4['month']       = df_q4['published_date'].str[:7]
    df_q4['engagements'] = df_q4['likes'] + df_q4['comments']
    df_q4['engagement_rate'] = (df_q4['engagements'] / df_q4['views'].replace(0,1) * 100).round(4)
    df_q4.to_csv(f'{DATA_DIR}/q4_2024_videos.csv', index=False)
    print(f'Saved {len(df_q4)} Q4 rows.')
else:
    df_q4 = pd.read_csv(f'{DATA_DIR}/q4_2024_videos.csv')
    print(f'Loaded {len(df_q4)} Q4 rows from file.')

df_q4.head(3)

## 4. Google Trends via pytrends

In [ ]:
if REFRESH_DATA:
    from pytrends.request import TrendReq
    pytrends = TrendReq(hl='en-US', tz=360)
    pytrends.build_payload(
        kw_list=['valorant'],
        cat=0,
        timeframe='2024-07-01 2024-12-31',
        geo='US',
        gprop=''
    )
    df_trends = pytrends.interest_over_time()
    if 'isPartial' in df_trends.columns:
        df_trends = df_trends.drop(columns=['isPartial'])
    df_trends = df_trends.reset_index().rename(columns={'date': 'week', 'valorant': 'interest'})
    df_trends['month'] = df_trends['week'].dt.to_period('M').astype(str)
    monthly = df_trends.groupby('month')['interest'].mean().round(1).reset_index()
    monthly.columns = ['month', 'avg_interest']
    monthly.to_csv(f'{DATA_DIR}/search_interest_monthly.csv', index=False)
    print('Trends saved.')
else:
    monthly = pd.read_csv(f'{DATA_DIR}/search_interest_monthly.csv')
    print('Loaded trends from file.')

print(monthly.to_string(index=False))

## 5. Validation Summary

In [ ]:
print('=== DATA VALIDATION SUMMARY ===')
print(f'Q3 videos : {len(df_q3)} rows | {df_q3.creator.nunique()} creators')
print(f'Q4 videos : {len(df_q4)} rows | {df_q4.creator.nunique()} creators')
print(f'Total     : {len(df_q3)+len(df_q4)} videos')
print()

# Check for zero-view rows
zero_views = (df_q3['views'] == 0).sum() + (df_q4['views'] == 0).sum()
print(f'Zero-view rows : {zero_views}')

# Per-creator video counts
counts = pd.concat([df_q3.groupby('creator').size().rename('Q3'),
                    df_q4.groupby('creator').size().rename('Q4')], axis=1).fillna(0).astype(int)
counts['Total'] = counts['Q3'] + counts['Q4']
print()
print(counts)

# TenZ viral outlier check
viral = df_q3[df_q3['views'] > 1_000_000]
print(f'\nVideos >1M views in Q3:')
print(viral[['creator','title','views','engagement_rate']].to_string(index=False))